# Chained analog agents — NL spec → topology → sizing → SPICE → corners

Deeper companion to `agents_miller_ota.ipynb`. Exercises the full analog chain:

1. `analog-topology-recommender` — natural-language spec → JSON topology choice.
2. `analog-sizing-advisor` — topology + gm/ID LUTs → starter sizing vector.
3. `AutoresearchRunner` — greedy SPICE refinement, budget=8.
4. `analog.corner_validator` — PVT sweep on the winner.

Companion to the deep deck at `../claude_design_slides/analog.html`.

All `RUN_*` flags default to `False` — opening the notebook is a safe no-op.

**Prerequisites** — see `./README.md`. Needs the `eda_agents` package (pip -e .),
ngspice, `PDK_ROOT`, `EDA_AGENTS_IHP_LUT_DIR`, plus either `OPENROUTER_API_KEY`
or `ZAI_API_KEY`.

## Step 0 — Editable install + environment check

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

RUN_PIP_INSTALL       = False
RUN_RECOMMENDER       = False  # real CLI call; leave False to use canned JSON
RUN_SIZING_ADVISOR    = False  # advisor is a thin wrapper; starter=default_params
RUN_AUTORESEARCH      = False  # flip after dry-run sanity check
RUN_CORNER_VALIDATOR  = False  # post-convergence
AUTORESEARCH_BUDGET   = 8
LLM_MODEL             = "zai/GLM-4.5-Flash"
SPEC                  = "Rail-to-rail OTA, 10 MHz GBW, 60 dB gain, 1 pF load, IHP SG13G2."

REPO_ROOT = Path.cwd().resolve().parents[2]
WORK_DIR  = Path.cwd() / "analog_chain_results"

print(f"Repo root              : {REPO_ROOT}")
print(f"venv                   : {os.environ.get('VIRTUAL_ENV', 'UNSET')}")
print(f"ngspice on PATH        : {shutil.which('ngspice') or 'MISSING'}")
print(f"PDK_ROOT               : {os.environ.get('PDK_ROOT', '') or 'UNSET'}")
print(f"EDA_AGENTS_IHP_LUT_DIR : {os.environ.get('EDA_AGENTS_IHP_LUT_DIR', '') or 'UNSET'}")
for k in ("OPENROUTER_API_KEY", "ZAI_API_KEY"):
    print(f"{k:<22}: {'set' if os.environ.get(k) else 'unset'}")

if RUN_PIP_INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], check=True)
else:
    print("[rehearse] RUN_PIP_INSTALL=False — skipping.")

## Step 1 — Recommender (natural-language spec → topology JSON)

The `analog-topology-recommender` agent takes a one-line spec and returns
a single JSON object with `topology`, `rationale`, `starter_specs`, and
`confidence`. We invoke it from a Claude Code / OpenCode CLI separately
(or via MCP) and paste the output here.

This cell uses a canned response by default so the chain runs without
needing an active CLI. Flip `RUN_RECOMMENDER=True` if you want to wire
it to your own invocation.

In [ ]:
import json

if RUN_RECOMMENDER:
    # Wire your CLI call here. e.g. subprocess.run(['opencode','run', ...])
    # and parse the JSON from stdout. Out of scope for this notebook.
    topology_rec = None
    raise NotImplementedError("plug your recommender invocation here")
else:
    topology_rec = {
        "topology":      "miller_ota",
        "rationale":     "Miller compensation gives a predictable roll-off at the requested frequencies.",
        "starter_specs": {"gbw_hz": 10e6, "pm_deg": 60, "cl_f": 1e-12, "pdk": "ihp_sg13g2"},
        "confidence":    0.82,
    }

print(f"spec      : {SPEC}")
print(f"topology  : {topology_rec['topology']}  (confidence {topology_rec['confidence']})")
print(f"rationale : {topology_rec['rationale']}")
print("starter_specs:")
print(json.dumps(topology_rec['starter_specs'], indent=2))

## Step 2 — Sizing advisor (topology + gm/ID LUTs → starter vector)

`analog-sizing-advisor` consumes the recommender's JSON and the gm/ID LUTs,
and emits a starting sizing vector. For this notebook we use `default_params()`
from the CircuitTopology as the seed — the advisor's refinement is a thin
wrapper around the same LUT lookup the topology already uses internally.

In [ ]:
if topology_rec['topology'] != 'miller_ota':
    raise RuntimeError(f"This notebook wires only miller_ota; recommender returned {topology_rec['topology']}")

from eda_agents.topologies.ota_miller import MillerOTATopology
topo = MillerOTATopology()
starter = topo.default_params()
print(f"topology_name : {topo.topology_name()}")
print(f"pdk           : {topo.pdk}")
print("starter params (gm/ID-seeded defaults):")
for k, v in starter.items():
    print(f"  {k} = {v}")

## Step 3 — One SPICE eval at the starter (sanity)

In [ ]:
import tempfile
from eda_agents.core.spice_runner import SpiceRunner

runner = SpiceRunner(pdk=topo.pdk)
missing = runner.validate_pdk()
print(f"PDK problems: {missing or 'none'}")

if not missing:
    sizing = topo.params_to_sizing(starter)
    cir    = topo.generate_netlist(sizing, Path(tempfile.mkdtemp()))
    result = runner.run(cir)
    if result.success:
        fom = topo.compute_fom(result, sizing)
        print(f"starter  Adc={result.Adc_dB:.1f} dB  GBW={result.GBW_Hz/1e6:.2f} MHz  PM={result.PM_deg:.1f} deg  FoM={fom:.3e}")
    else:
        print(f"SPICE failed: {result.error}")

## Step 4 — AutoresearchRunner — the greedy loop (budget 8)

In [ ]:
import asyncio
from eda_agents.agents.autoresearch_runner import AutoresearchRunner

async def _run():
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    r = AutoresearchRunner(topology=topo, model=LLM_MODEL, budget=AUTORESEARCH_BUDGET)
    return await r.run(WORK_DIR)

if RUN_AUTORESEARCH:
    result = asyncio.get_event_loop().run_until_complete(_run())
    print(result.summary)
    print(f"validity_rate : {result.validity_rate:.0%}")
    if result.best_valid:
        print("best_params   :", json.dumps(result.best_params, indent=2))
else:
    print("[rehearse] RUN_AUTORESEARCH=False — skipping real loop.")
    print(f"When enabled: program.md + results.tsv land under {WORK_DIR}")

## Step 5 — Corner validator (post-convergence PVT sweep)

`analog.corner_validator` re-runs the winning sizing across process / voltage
/ temperature corners. On failure, the runner backs off to the previous
candidate (row *n* − 1 in `results.tsv`) and re-validates.

This is wired as a CLI-invoked skill today; the cell below prints the command
you would run once autoresearch has converged.

In [ ]:
if RUN_CORNER_VALIDATOR:
    print("Invoke the corner validator through your CLI of choice; this cell is a stub.")
else:
    print("Typical corner matrix  : 5 process x 3 voltage x 3 temperature = 45 re-runs.")
    print("Expected wall time     : ~8 minutes for a Miller OTA at typical ngspice speed.")
    print("On failure             : runner backs off one row in results.tsv and retries.")

## Step 6 — Tail `program.md`

In [ ]:
prog = WORK_DIR / "program.md"
tsv  = WORK_DIR / "results.tsv"
if prog.exists():
    print(prog.read_text()[:2500])
else:
    print(f"{prog} not yet written; flip RUN_AUTORESEARCH=True first.")
if tsv.exists():
    rows = tsv.read_text().splitlines()
    print("\n--- results.tsv (head + tail) ---")
    for r in rows[:1] + rows[-8:]:
        print(r)

## Cleanup

```bash
rm -rf analog_chain_results/
```

## Next

- Short analog loop: `./agents_miller_ota.ipynb`.
- Digital companion: `./agents_rtl2gds_counter.ipynb` and
  `./agents_digital_autoresearch.ipynb`.
- Slide deck: `../claude_design_slides/analog.html`.